# Applied Computer Vision - Densidade Urbana por Imagem de Satelite

Notebook de treinamento, avaliacao e inferencia das CNNs usadas pelo UrbanLens Mobility.

Objetivo: classificar imagens reais de satelite nas classes `baixa`, `media` e `alta` densidade urbana visual.

## 1. Configuracao do ambiente

O codigo abaixo funciona com o notebook aberto pela raiz do projeto ou diretamente pela pasta `notebooks`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / 'src').is_dir() else CURRENT_DIR.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifest.csv'
COMPARISON_PATH = PROJECT_ROOT / 'reports' / 'model_comparison.csv'
BEST_METADATA_PATH = PROJECT_ROOT / 'models' / 'best_model_metadata.json'

assert SRC_DIR.is_dir(), f'Pasta src nao encontrada em {PROJECT_ROOT}'
print(f'Projeto: {PROJECT_ROOT}')

## 2. Preparacao do dataset

A preparacao baixa ou reutiliza os recortes do ArcGIS World Imagery, calcula atributos visuais e preserva os splits existentes. Deixe a flag como `False` quando os arquivos ja estiverem prontos.

In [ ]:
RUN_DATA_PREPARATION = False

if RUN_DATA_PREPARATION:
    subprocess.run(
        [sys.executable, str(SRC_DIR / 'prepare_dataset.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Preparacao ignorada: usando o dataset versionado no repositorio.')

## 3. Distribuicao e amostras

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
distribution = (
    manifest.groupby(['split', 'visual_density_class'])
    .size()
    .rename('images')
    .reset_index()
)
display(distribution)
print(f'Total de imagens: {len(manifest)}')

In [ ]:
samples = manifest.groupby('visual_density_class').head(2).reset_index(drop=True)
figure, axes = plt.subplots(2, 3, figsize=(13, 8))
for axis, (_, row) in zip(axes.flat, samples.iterrows()):
    image_path = PROJECT_ROOT / Path(row['local_image_path'])
    axis.imshow(Image.open(image_path).convert('RGB'))
    axis.set_title(f"{row['visual_density_class']} | {row['name']}")
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. Arquiteturas

As quatro CNNs sao implementadas do zero em `src/models.py`. A especificacao detalhada tambem esta em `ARQUITETURA_MODELOS.md` e nos arquivos `reports/model_architectures.*`.

In [ ]:
from models import build_model, count_parameters

MODEL_NAMES = ['urban_cnn_v1', 'urban_cnn_v2', 'urban_cnn_v3', 'urban_cnn_v4']
architectures = []
for model_name in MODEL_NAMES:
    model = build_model(model_name)
    architectures.append(
        {'modelo': model_name, 'parametros_treinaveis': count_parameters(model)}
    )
display(pd.DataFrame(architectures))
print(build_model('urban_cnn_v1'))

## 5. Treinamento

O script aplica normalizacao calculada no treino, data augmentation, `CrossEntropyLoss`, `AdamW`, scheduler, early stopping e avaliacao final. Ative a flag para refazer o treinamento completo.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [sys.executable, str(SRC_DIR / 'train.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Treinamento ignorado: usando os pesos e relatorios versionados.')

## 6. Comparacao e desempenho

In [ ]:
comparison = pd.read_csv(COMPARISON_PATH)
display(comparison)

best_metadata = json.loads(BEST_METADATA_PATH.read_text(encoding='utf-8'))
print(f"Melhor modelo: {best_metadata['best_model']}")
print(f"Acuracia em teste: {best_metadata['test_accuracy']:.2%}")

In [ ]:
best_model = best_metadata['best_model']
artifact_paths = [
    PROJECT_ROOT / 'reports' / f'training_curves_{best_model}.png',
    PROJECT_ROOT / 'reports' / f'confusion_matrix_{best_model}.png',
    PROJECT_ROOT / 'reports' / f'error_examples_{best_model}.png',
]

for artifact_path in artifact_paths:
    if artifact_path.exists():
        display(Image.open(artifact_path))

## 7. Inferencia com o melhor modelo

In [ ]:
from inference import predict_image

sample_row = manifest.loc[manifest['split'] == 'test'].iloc[0]
sample_path = PROJECT_ROOT / Path(sample_row['local_image_path'])
sample_image = Image.open(sample_path).convert('RGB')
prediction = predict_image(sample_image)

display(sample_image)
print(f"Rotulo real: {sample_row['visual_density_class']}")
print(f"Classe prevista: {prediction['predicted_class']}")
display(pd.Series(prediction['probabilities'], name='probabilidade').to_frame())

## Resultado

O melhor modelo e o `UrbanDensityCNNV1`, com 88,89% de acuracia no conjunto de teste (8 acertos em 9 imagens). Como o conjunto de teste e pequeno, a metrica deve ser tratada como resultado preliminar e complementada por novas amostras.